# 02 — Feature Engineering

This notebook builds the labeled feature matrix from raw on-chain data.

**Pipeline:**
1. Load raw transaction data (normal txs + ERC-20 transfers)
2. Load labels (default cohort from Aave V2 liquidations, non-default cohort)
3. Build all 45 features across 5 families
4. Analyze feature distributions and check for data quality issues
5. Save the feature matrix for training

**Feature families:**
| Family | Count | Description |
|---|---|---|
| Transaction Volume | 9 | Raw activity and ETH flow |
| Counterparty Graph | 7 | Diversity and concentration |
| Protocol Diversity | 7 | DeFi breadth and engagement |
| Collateral Behavior | 8 | Aave-specific interactions and gas patterns |
| Temporal Consistency | 14 | Activity regularity over time |

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.builder import build_feature_matrix

sns.set_theme(style='whitegrid', context='notebook', palette='muted')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

BRAND_BLUE = '#185FA5'

## 1. Build feature matrix

This cell runs the full feature pipeline. If the feature matrix already exists, skip to cell 3.

In [ ]:
FEATURE_PATH = Path('../data/processed/feature_matrix.parquet')

if not FEATURE_PATH.exists():
    print('Building feature matrix...')
    df = build_feature_matrix(
        normal_txs_path=Path('../data/raw/wallets/normal_txs.parquet'),
        token_txs_path=Path('../data/raw/wallets/token_txs.parquet'),
        labeled_default_path=Path('../data/raw/aave_v2_liquidations.parquet'),
        labeled_nondefault_path=Path('../data/raw/non_default_cohort.parquet'),
        output_path=FEATURE_PATH,
    )
else:
    print('Feature matrix already exists, loading...')
    df = pd.read_parquet(FEATURE_PATH)

print(f'Shape: {df.shape}')
print(f'Default rate: {df["label"].mean():.2%}')
df.head(3)

## 2. Class balance

Credit default datasets are inherently imbalanced — real default rates are low.
This is handled in training via `class_weight='balanced'` and `scale_pos_weight`.

In [ ]:
label_counts = df['label'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind='bar', ax=ax, color=[BRAND_BLUE, '#E05C2A'], edgecolor='white')
ax.set_xticklabels(['Non-default (0)', 'Default (1)'], rotation=0)
ax.set_title('Class balance')
ax.set_ylabel('Number of wallets')
for i, v in enumerate(label_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Imbalance ratio (non-default:default): {label_counts[0]/label_counts[1]:.1f}:1')

## 3. Feature distributions by class

Visualize the most important features separated by class.
Features with good class separation will be valuable predictors.

In [ ]:
key_features = [
    'tx_count', 'wallet_age_days', 'repay_to_borrow_ratio',
    'protocols_used_count', 'avg_days_between_txs', 'contract_ratio',
    'aave_borrow_count', 'recent_tx_ratio', 'unique_counterparties',
]

# Only plot features that exist
key_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    for label, color, name in [(0, BRAND_BLUE, 'Non-default'), (1, '#E05C2A', 'Default')]:
        vals = df.loc[df['label'] == label, feat].clip(
            upper=df[feat].quantile(0.99)
        )
        ax.hist(vals, bins=30, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', y=1.01, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation matrix

Check for highly correlated features (> 0.9) that could cause multicollinearity issues in
Logistic Regression. LightGBM is robust to this, but LR may need feature selection.

In [ ]:
feat_cols = [c for c in df.columns if c not in ('wallet', 'label', 'first_tx_block', 'last_tx_block')]
corr = df[feat_cols].corr()

# Find pairs with |r| > 0.85
high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr.columns = ['feature_1', 'feature_2', 'r']
high_corr = high_corr[high_corr['r'].abs() > 0.85].sort_values('r', ascending=False)
print('Highly correlated feature pairs (|r| > 0.85):')
print(high_corr.to_string(index=False))

# Heatmap of top 20 features only
top20 = feat_cols[:20]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    df[top20].corr(), annot=False, cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, ax=ax, linewidths=0.3
)
ax.set_title('Correlation Heatmap (top 20 features)')
plt.tight_layout()
plt.show()

## 5. Point-biserial correlations with label

Quick univariate screening: which individual features correlate most with the default label?

In [ ]:
from scipy.stats import pointbiserialr

corr_with_label = []
for feat in feat_cols:
    r, p = pointbiserialr(df['label'], df[feat].fillna(0))
    corr_with_label.append({'feature': feat, 'correlation': r, 'p_value': p})

corr_df = pd.DataFrame(corr_with_label).sort_values('correlation', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
top20_corr = corr_df.head(20)
colors = ['#E05C2A' if r > 0 else BRAND_BLUE for r in top20_corr['correlation']]
ax.barh(top20_corr['feature'], top20_corr['correlation'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Top 20 Features by Correlation with Default Label')
ax.set_xlabel('Point-biserial correlation (positive = higher risk)')
plt.tight_layout()
plt.show()

## 6. Missing values and data quality

In [ ]:
missing = df[feat_cols].isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
print('Missing value percentage per feature:')
print(missing_pct[missing_pct > 0].to_string())
if missing_pct.max() == 0:
    print('No missing values — feature matrix is complete.')

## Next step

Feature matrix looks good. Proceed to `03_model_training.ipynb` to train and evaluate the models.